# Lab 1: Shipment Logistics Analysis
## Set up storage

Create the personal schema and volume, and define paths for the source dataset.

In [0]:
CATALOG = "dbr_dev_ua5816bd"
SCHEMA = "mialkovska_viktor594"
VOLUME = "raw_files"

RAW_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
DATASET_PATH = f"{RAW_PATH}/smart_shipment_route_monitoring"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

## Define source path

Set the path to the shipment CSV stored in the Unity Catalog volume.

In [0]:
DATA_PATH = (
    f"/Volumes/{CATALOG}/{SCHEMA}/raw_files/"
    "smart_shipment_route_monitoring/shipment"
)

SHIPMENTS_PATH = f"{DATA_PATH}/shipments_master.csv"

print(SHIPMENTS_PATH)

## Load shipment data

Read the shipment dataset into a Spark DataFrame and preview sample records.

In [0]:
shipments_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(SHIPMENTS_PATH)
)


display(shipments_df.limit(10))

## Inspect schema

Review the inferred data types of the shipment dataset.

In [0]:
shipments_df.printSchema()


## Save raw shipment table

Write the source shipment data to a Delta table in the personal schema.

In [0]:
SHIPMENTS_RAW = f"{CATALOG}.{SCHEMA}.shipments_raw"

(
    shipments_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(SHIPMENTS_RAW)
)


## Prepare shipment locations

Extract distinct origin and destination locations for API enrichment.

In [0]:
shipments_df = spark.table(SHIPMENTS_RAW)

origins = shipments_df.select("origin_port")
destinations = shipments_df.select("destination_port")

locations_df = (
    origins
    .withColumnRenamed("origin_port", "location_name")
    .union(
        destinations.withColumnRenamed("destination_port", "location_name")
    )
    .dropna()
    .distinct()
)

print(f"Distinct locations: {locations_df.count()}")

display(locations_df)

## Enrich locations with external API

To meet the lab requirement of using an external API, I used the Open-Meteo Geocoding API to enrich shipment locations with country and country code information.

In [0]:
import requests

def get_location_info(location_name):

    try:
        response = requests.get(
            "https://geocoding-api.open-meteo.com/v1/search",
            params={"name": location_name.replace("_", " "), "count": 1},
            timeout=20
        )

        data = response.json()

        if "results" not in data:
            return {
                "location_name": location_name,
                "country": None,
                "country_code": None
            }

        result = data["results"][0]

        return {
            "location_name": location_name,
            "country": result.get("country"),
            "country_code": result.get("country_code")
        }

    except Exception as e:
        print("Failed:", location_name, e)

        return {
            "location_name": location_name,
            "country": None,
            "country_code": None
        }

### Fetch location data

Call the geocoding API for each distinct shipment location and collect the enrichment results

In [0]:
import time

locations = [
    row["location_name"]
    for row in locations_df.collect()
]

location_results = []

for i, location in enumerate(locations, start=1):
    
    result = get_location_info(location)
    location_results.append(result)
    
    print(f"{location} -> {result['country']}")
    
    time.sleep(0.2)

### Create location DataFrame

Convert the API results into a Spark DataFrame with a defined schema

In [0]:
from pyspark.sql.types import StringType, StructField, StructType

location_schema = StructType([
    StructField("location_name", StringType(), False),
    StructField("country", StringType(), True),
    StructField("country_code", StringType(), True)
])

location_df = spark.createDataFrame(
    location_results,
    schema=location_schema
)

display(location_df)


### Prepare location lookups

Create separate origin and destination lookup DataFrames for the following joins.

In [0]:
from pyspark.sql import functions as F

origin_df = location_df.select(
    F.col("location_name").alias("origin_port"),
    F.col("country").alias("origin_country")
)

destination_df = location_df.select(
    F.col("location_name").alias("destination_port"),
    F.col("country").alias("destination_country")
)

## Enrich shipment data

Join shipment records with origin and destination country information.

In [0]:
shipment_analysis_df = (
    shipments_df
    .join(origin_df, on="origin_port", how="left")
    .join(destination_df, on="destination_port", how="left")
)

shipment_analysis_df.printSchema()

## Select analysis fields

Keep only the columns required for further analysis and dashboarding.

In [0]:
shipment_analysis_df = shipment_analysis_df.select(
    "shipment_id",
    "carrier",
    "origin_port",
    "origin_country",
    "destination_port",
    "destination_country",
    "transport_mode",
    "status",
    "distance_km",
    "freight_cost_usd",
    "delay_hours",
    "risk_score"
)

display(shipment_analysis_df.limit(10))

## Clean status values

Replace underscores in shipment statuses

In [0]:

shipment_analysis_df = shipment_analysis_df.withColumn(
    "status",
    F.regexp_replace("status", "_", " ")
)

## Save final analysis table

Write the prepared dataset to a Delta table for analytics and dashboard use

In [0]:
FINAL_TABLE = f"{CATALOG}.{SCHEMA}.shipment_analysis"

(
    shipment_analysis_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(FINAL_TABLE)
)

print(f"Created: {FINAL_TABLE}")

## Analyze shipment status

Count shipments by status to identify the most common shipment states.

In [0]:
status_analysis_df = (
    shipment_analysis_df
    .groupBy("status")
    .agg(F.count("*").alias("shipments"))
    .orderBy(F.desc("shipments"))
)

display(status_analysis_df)

## Analyze delays by transport mode

Compare average shipment delay across different transport modes.

In [0]:
transport_delay_df = (
    shipment_analysis_df
    .groupBy("transport_mode")
    .agg(
        F.round(F.avg("delay_hours"), 2)
        .alias("avg_delay_hours")
    )
    .orderBy(F.desc("avg_delay_hours"))
)

display(transport_delay_df)

## Identify high-risk shipments

Display the top 10 shipments with the highest risk scores

In [0]:
top_risk_shipments_df = (
    shipment_analysis_df
    .select(
        "shipment_id",
        "carrier",
        "origin_country",
        "destination_country",
        "delay_hours",
        "risk_score"
    )
    .orderBy(F.desc("risk_score"))
    .limit(10)
)

display(top_risk_shipments_df)